In [19]:
# === 1. PENGUMPULAN DATA ===
import pandas as pd

# Load dataset
df = pd.read_csv("dataset_chatbot.csv")

print("Jumlah data:", len(df))
df.head()


Jumlah data: 250


,pertanyaan,jawaban,intent
0,Bisakah daftar kuliah secara online?,informasi lengkap seputar pendaftran dapat dil...,pendaftaran
1,Apa betul ada jalur beasiswa saat pendaftaran?,informasi lengkap seputar pendaftran dapat dil...,pendaftaran
2,Apakah ada beasiswa untuk mahasiswa berprestasi?,informasi terkait beasiswa dapat dilihat melal...,administrasi
3,Apakah UAS perlu cetak kartu ujian?,kartu ujian syarat wajib untuk bisa mengikuti ...,ujian
4,Apakah ada jadwal kuliah malam?,tidak ada jadwal kuliah malam,lainnya


In [21]:
# === 2. PRAPROSES TEKS ===
import re
import nltk
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

nltk.download('punkt')
nltk.download('punkt_tab')   # untuk versi terbaru
nltk.download('stopwords')


# Stopwords bahasa Indonesia
stop_words = set(stopwords.words('indonesian'))

# Stemming
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def preprocess(text):
    # Case folding
    text = text.lower()
    # Hapus karakter selain huruf
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Tokenisasi
    tokens = nltk.word_tokenize(text)
    # Stopword removal + stemming
    tokens = [stemmer.stem(word) for word in tokens if word not in stop_words]
    return " ".join(tokens)

df['clean_text'] = df['pertanyaan'].apply(preprocess)
df[['pertanyaan', 'clean_text']].head(10)


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\fadhl\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\fadhl\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\fadhl\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,pertanyaan,clean_text
0,Bisakah daftar kuliah secara online?,daftar kuliah online
1,Apa betul ada jalur beasiswa saat pendaftaran?,jalur beasiswa daftar
2,Apakah ada beasiswa untuk mahasiswa berprestasi?,beasiswa mahasiswa prestasi
3,Apakah UAS perlu cetak kartu ujian?,uas cetak kartu uji
4,Apakah ada jadwal kuliah malam?,jadwal kuliah malam
5,Dimana lokasi ujian dilaksanakan?,mana lokasi uji laksana
6,Apa betul ada biaya bimbingan skripsi?,biaya bimbing skripsi
7,Apakah pengisian KRS bisa dilakukan secara onl...,isi krs online
8,Dimana lokasi pendaftaran mahasiswa baru?,mana lokasi daftar mahasiswa
9,Apa betul ada grup WhatsApp untuk mahasiswa?,grup whatsapp mahasiswa


In [23]:
import pandas as pd
import re, string
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Ambil beberapa contoh pertanyaan (misalnya 5 baris pertama)
sample_data = df['pertanyaan'].iloc[:5]

# Stopwords
stop_words = set(stopwords.words('indonesian'))

# Stemmer Indonesia
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Simpan hasil praproses
results = []

for teks in sample_data:
    row = {}
    row["Teks Asli"] = teks

    # 1. Case Folding
    step1 = teks.lower()
    row["Case Folding"] = step1

    # 2. Hapus tanda baca
    step2 = re.sub(f"[{string.punctuation}]", " ", step1)

    # 3. Tokenisasi
    step3 = step2.split()
    row["Tokenisasi"] = step3

    # 4. Stopword Removal
    step4 = [word for word in step3 if word not in stop_words]
    row["Stopword Removal"] = step4

    # 5. Stemming (pakai Sastrawi)
    step5 = [stemmer.stem(word) for word in step4]
    row["Stemming"] = step5

    # Hasil akhir
    row["Hasil Akhir"] = " ".join(step5)

    results.append(row)

# DataFrame hasil
df_process = pd.DataFrame(results)
print(df_process)


                                          Teks Asli  \
0              Bisakah daftar kuliah secara online?   
1    Apa betul ada jalur beasiswa saat pendaftaran?   
2  Apakah ada beasiswa untuk mahasiswa berprestasi?   
3               Apakah UAS perlu cetak kartu ujian?   
4                   Apakah ada jadwal kuliah malam?   

                                       Case Folding  \
0              bisakah daftar kuliah secara online?   
1    apa betul ada jalur beasiswa saat pendaftaran?   
2  apakah ada beasiswa untuk mahasiswa berprestasi?   
3               apakah uas perlu cetak kartu ujian?   
4                   apakah ada jadwal kuliah malam?   

                                          Tokenisasi  \
0          [bisakah, daftar, kuliah, secara, online]   
1  [apa, betul, ada, jalur, beasiswa, saat, penda...   
2  [apakah, ada, beasiswa, untuk, mahasiswa, berp...   
3          [apakah, uas, perlu, cetak, kartu, ujian]   
4               [apakah, ada, jadwal, kuliah, malam]   

 

In [25]:
# === 3. EKSTRAKSI FITUR (TF-IDF) ===
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['clean_text'])
y = df['intent']

print("Shape fitur:", X.shape)


Shape fitur: (250, 111)


In [47]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import pandas as pd

# Ambil teks bersih (clean_text) dari dataset
X_clean = df['clean_text']

# === Gunakan TF-IDF ===
vectorizer = TfidfVectorizer()
X_features = vectorizer.fit_transform(X_clean)

# Ambil daftar fitur (kata-kata)
feature_names = vectorizer.get_feature_names_out()

# Ubah ke DataFrame biar lebih mudah dilihat
df_features = pd.DataFrame(
    X_features.toarray(),
    columns=feature_names
)

# Tampilkan 5 dokumen pertama dan 10 fitur pertama
print(df_features.iloc[:15, :10])


    ada  administrasi       aju  akademik  aktif  aktivasi  akun     ambil  \
0   0.0           0.0  0.000000       0.0    0.0       0.0   0.0  0.000000   
1   0.0           0.0  0.000000       0.0    0.0       0.0   0.0  0.000000   
2   0.0           0.0  0.000000       0.0    0.0       0.0   0.0  0.000000   
3   0.0           0.0  0.000000       0.0    0.0       0.0   0.0  0.000000   
4   0.0           0.0  0.000000       0.0    0.0       0.0   0.0  0.000000   
5   0.0           0.0  0.000000       0.0    0.0       0.0   0.0  0.000000   
6   0.0           0.0  0.000000       0.0    0.0       0.0   0.0  0.000000   
7   0.0           0.0  0.000000       0.0    0.0       0.0   0.0  0.000000   
8   0.0           0.0  0.000000       0.0    0.0       0.0   0.0  0.000000   
9   0.0           0.0  0.000000       0.0    0.0       0.0   0.0  0.000000   
10  0.0           0.0  0.000000       0.0    0.0       0.0   0.0  0.000000   
11  0.0           0.0  0.000000       0.0    0.0       0.0   0.0

In [29]:
# === 4. PELATIHAN MODEL (Naive Bayes) ===
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model Naive Bayes
model = MultinomialNB()
model.fit(X_train, y_train)

# Prediksi
y_pred = model.predict(X_test)


In [15]:
# === 5. EVALUASI MODEL ===
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

print("Akurasi:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average='weighted'))
print("Recall:", recall_score(y_test, y_pred, average='weighted'))
print("F1-Score:", f1_score(y_test, y_pred, average='weighted'))
print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred))


Akurasi: 0.96
Precision: 0.9488888888888889
Recall: 0.96
F1-Score: 0.9510714285714286

=== Classification Report ===
              precision    recall  f1-score   support

administrasi       1.00      1.00      1.00         5
    beasiswa       0.00      0.00      0.00         1
         krs       1.00      1.00      1.00        13
     lainnya       1.00      0.75      0.86         4
 pendaftaran       0.78      1.00      0.88         7
     skripsi       1.00      1.00      1.00        13
       ujian       1.00      1.00      1.00         7

    accuracy                           0.96        50
   macro avg       0.83      0.82      0.82        50
weighted avg       0.95      0.96      0.95        50



C:\anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1531: Undefine

In [51]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# === 6. IMPLEMENTASI DAN EVALUASI (Chatbot Lebih Fleksibel) ===
# Simpan semua pertanyaan & jawaban dari dataset
corpus_vec = vectorizer.transform(df['clean_text']).toarray()
corpus_texts = df['pertanyaan'].tolist()
corpus_answers = df['jawaban'].tolist()

# Fungsi chatbot
def chatbot_response(text, threshold=0.4):
    # Preprocessing
    clean = preprocess(text)
    vec = vectorizer.transform([clean])
    
    # Prediksi intent (Naive Bayes)
    pred_intent = model.predict(vec)[0]
    pred_prob = np.max(model.predict_proba(vec))  # confidence
    
    if pred_prob >= threshold:
        # Cari jawaban langsung dari dataset sesuai intent
        candidates = df[df['intent'] == pred_intent]['jawaban'].tolist()
        response = np.random.choice(candidates) if candidates else "Maaf, saya belum punya jawaban untuk intent ini."
    else:
        # Kalau confidence rendah → cari jawaban paling mirip dari data training
        sim_scores = cosine_similarity(vec.toarray(), corpus_vec)[0]
        idx = np.argmax(sim_scores)
        response = corpus_answers[idx]
    
    return response

# Uji chatbot interaktif
print("=== Chatbot Siap ===")
while True:
    user = input("Anda: ")
    if user.lower() in ["exit", "quit", "keluar"]:
        print("Chatbot: Terima kasih, sampai jumpa!")
        break
    print("Chatbot:", chatbot_response(user))

=== Chatbot Siap ===


Anda:  cara melihat nilai?


Chatbot: cara mengajukan keberatan nilai silahkan menghubungi operator siakad


Anda:  bagaimana berkomunikasi dengan dosen wali?


Chatbot: dapat menghubungi langsung melalui pesan whatsapp


Anda:  quit


Chatbot: Terima kasih, sampai jumpa!


In [49]:
import joblib

# Simpan model dan vectorizer
joblib.dump(model, "naive_bayes_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

['tfidf_vectorizer.pkl']